Setup and Dataset Loading

In [ ]:
# Install required libraries
!pip install datasets sentence-transformers pandas numpy -q
from datasets import load_dataset
import pandas as pd
import numpy as np
import os
# Load the ArXiv dataset
dataset = load_dataset("CShorten/ML-ArXiv-Papers")
df = dataset["train"].to_pandas()
print(f"Dataset Loaded. Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Dataset Loaded. Shape: (117592, 4)
Columns: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract']


Data Cleaning & Preprocessing

In [ ]:
# Select only title and abstract
df = df[['title', 'abstract']]
# Combine Title and Abstract for full context
df['paper_text'] = df['title'] + " " + df['abstract']
# Clean structural garbage (newlines and extra spaces)
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()
# Use a subset for faster processing
df = df.head(15000)
print("Preprocessing Complete.")
print(f"Sample Processed Text: {df['paper_text'].iloc[0][:200]}...")

Preprocessing Complete.
Sample Processed Text: Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d...


Loading the Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer
# Load the pre-trained model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Model Loaded Successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded Successfully.


Generating and Saving Embeddings

In [ ]:
# Create a data directory to store artifacts
os.makedirs('data', exist_ok=True)
print("Generating Embeddings (this may take time)...")
# We encode the text in batches for efficiency
embeddings = model.encode(
    df["paper_text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)
# Save the embeddings and the cleaned dataframe
np.save("data/arxiv_embeddings.npy", embeddings)
df.to_csv("data/cleaned_arxiv_papers.csv", index=False)

print("All artifacts saved to /data folder.")

Generating Embeddings (this may take time)...


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

All artifacts saved to /data folder.


Environment Setup and Data Loading

In [ ]:
!pip install faiss-cpu sentence-transformers pandas numpy -q
import faiss
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
import torch
# Load the saved data and embeddings
df = pd.read_csv("data/cleaned_arxiv_papers.csv")
embeddings = np.load("data/arxiv_embeddings.npy").astype('float32')
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(f"System Ready: {len(df)} research papers loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 55.2 MB/s eta 0:00:00


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

System Ready: 15000 research papers loaded.


Building the FAISS Index

In [ ]:
# L2 Normalization is required for Cosine Similarity in FAISS
faiss.normalize_L2(embeddings)
# Create the IndexFlatIP (Inner Product) index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS Index successfully built and ready for queries.")

FAISS Index successfully built and ready for queries.


In [ ]:
def search_paper(query, k=5):
    #Convert query to vector
    query_embedding = model.encode([query]).astype('float32')
    #Normalize query vector
    faiss.normalize_L2(query_embedding)
    #Search the index
    Q, A = index.search(query_embedding, k)
    #Display results
    for score, idx in zip(Q[0], A[0]):
        print(f"Similarity Score: {score:.4f}")
        print(f"Title: {df.iloc[idx]['title']}")
        print(f"Abstract Preview: {df.iloc[idx]['abstract'][:300]}...")
        print("-" * 50)
    return Q, A
# Test a search
search_paper("deep learning in business management")

Similarity Score: 0.7308
Title: Predicting Process Behaviour using Deep Learning
Abstract Preview:   Predicting business process behaviour is an important aspect of business
process management. Motivated by research in natural language processing, this
paper describes an application of deep learning with recurrent neural networks
to the problem of predicting the next event in a business process. ...
--------------------------------------------------
Similarity Score: 0.6323
Title: DeepAR: Probabilistic Forecasting with Autoregressive Recurrent Networks
Abstract Preview:   Probabilistic forecasting, i.e. estimating the probability distribution of a
time series' future given its past, is a key enabler for optimizing business
processes. In retail businesses, for example, forecasting demand is crucial for
having the right inventory available at the right time at the ri...
--------------------------------------------------
Similarity Score: 0.6263
Title: Improving Decision Analytics with De

(array([[0.73083997, 0.63233876, 0.6263274 , 0.6142603 , 0.6075958 ]],
       dtype=float32),
 array([[11623, 12992,  7072, 12572,  8580]]))

AI Summarization

In [ ]:
# Load the summarization model
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = 'cuda' if torch.cuda.is_available() else "cpu"
model_t5.to(device)
def generate_summary(text):
    input_text = "summarize: " + text
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    summary_ids = model_t5.generate(inputs, max_length=100, min_length=30, length_penalty=2.0, num_beams=4, early_stopping=True)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)
def searching_n_summarizing(query, k=3):
    Q, A = index.search(model.encode([query]).astype('float32'), k)
    print(f"AI RESEARCH BRIEF FOR: '{query}'")
    for score, idx in zip(Q[0], A[0]):
        print(f"TITLE: {df.iloc[idx]['title']}")
        summary = generate_summary(df.iloc[idx]['abstract'])
        print(f"AI SUMMARY: {summary}")
# Final integrated test
searching_n_summarizing("basics of natural language processing")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

AI RESEARCH BRIEF FOR: 'basics of natural language processing'
TITLE: Category-Theoretic Quantitative Compositional Distributional Models of
  Natural Language Semantics
AI SUMMARY: this thesis focuses on a particular approach to this compositionality problem. it combines syntactic analysis formalisms with distributional semantic representations of meaning to produce syntactically motivated composition operations. this approach can be theoretically extended and practically implemented to produce concrete compositional distributional models of natural language semantics.
TITLE: A Probabilistic Generative Grammar for Semantic Parsing
AI SUMMARY: domain-general semantic parsing is a long-standing goal in natural language processing. current approaches rely on additional supervision from new domains in order to generalize to those domains. our approach relies on domain-independent supervision to generalize to new domains.
TITLE: Natural Language Processing (almost) from Scratch
AI SUMMARY: